In [1]:
import numpy as np
import pandas as pd
import scipy.sparse
import joblib
import json
import implicit
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

### Import Data

In [2]:
item_df = pd.read_parquet("../data/item.parquet")
user_item_itera = pd.read_parquet("../data/user-item-interaction.parquet")

### Pre-processing

In [3]:
# Select relevant columns
merged_df = (
    user_item_itera[[
        "user_id", "parent_asin", "helpful_vote", "recency_weight", "review_word_count", "num_review_img", "review_rating"
    ]]
    .merge(
        item_df[["parent_asin", "num_item_img", "num_item_videos", "price"]],
        how="left",
        on="parent_asin"
    )
)

# Normalize feature columns (Min-Max scaling)
cols_to_norm = [
    col for col in merged_df.columns
    if col not in ["user_id", "parent_asin"]
]

scaled = scaler.fit_transform(merged_df[cols_to_norm])
merged_df[cols_to_norm] = scaled

# Invert price explicitly inside feature list
merged_df["price"] = 1 - merged_df["price"]

# Compute interaction score (unweighted mean, each feature contributes equally)
merged_df["interaction"] = merged_df[cols_to_norm].mean(axis=1)

# Aggregate to single score per (user, item)
merged_df = (
    merged_df
    .groupby(["user_id","parent_asin"], as_index=False)["interaction"].mean()
)

merged_df

,user_id,parent_asin,interaction
0,ae2222frpdmnomyomcwiantxp7uq,b002zf31nq,0.275388
1,ae22232ob6s7uac75jufyrgbshiq,b00ig2dokm,0.269761
2,ae22236afrrsmqikgg7tptb75qea,b006cq8tc2,0.265416
3,ae22236afrrsmqikgg7tptb75qea,b00b2v66vs,0.141672
4,ae22236afrrsmqikgg7tptb75qea,b00b7s5fdg,0.255419
...,...,...,...
4828475,ahzzzy4dflawpbqyfqfwvacngura,b009hkl4b8,0.265977
4828476,ahzzzy4dflawpbqyfqfwvacngura,b00ab7hesi,0.222515
4828477,ahzzzy4dflawpbqyfqfwvacngura,b014rgfc0k,0.235818
4828478,ahzzzy4dflawpbqyfqfwvacngura,b017s11xmc,0.268066


In [4]:
# Create categorical codes
user_cats = merged_df["user_id"].astype("category")
item_cats = merged_df["parent_asin"].astype("category")

merged_df["user_idx"] = user_cats.cat.codes
merged_df["item_idx"] = item_cats.cat.codes

# Mapping dictionaries
idx_to_userid = dict(enumerate(user_cats.cat.categories))
idx_to_itempasin = dict(enumerate(item_cats.cat.categories))

# Sparse matrix
num_users = len(user_cats.cat.categories)
num_items = len(item_cats.cat.categories)

interaction_matrix = scipy.sparse.csr_matrix(
    (
        merged_df["interaction"],
        (merged_df["user_idx"], merged_df["item_idx"])
    ),
    shape=(num_users, num_items)
).astype("float32")

interaction_matrix

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 4828480 stored elements and shape (2589466, 89246)>

### Model Training

In [5]:
# Train ALS
model = implicit.als.AlternatingLeastSquares(factors=50, regularization=0.01, iterations=20)
model.fit(interaction_matrix)  # transpose: ALS expects item-user

c:\Users\user\OneDrive\Desktop\Amozon\amozon_backend\.venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [6]:
# Get the *categorical code* for the user
target_idx = merged_df["user_idx"].iloc[0]

# Recommend
recomm_idx, recomm_scores = model.recommend(
    userid=target_idx,
    user_items=interaction_matrix[target_idx],
    N=5
)

recomm_asins = [idx_to_itempasin[i] for i in recomm_idx]

target_userid = idx_to_userid[target_idx]

print(f"Recommendations for {target_userid}")

recomm_items = item_df[item_df["parent_asin"].isin(recomm_asins)]

recomm_items[["parent_asin", "item_title", "main_category", "categories", "description", "features", "details"]]

Recommendations for ae2222frpdmnomyomcwiantxp7uq


,parent_asin,item_title,main_category,categories,description,features,details
3633,b00uc7dg6q,kindle for mac [download],software,"education & reference,software,writing & liter...",Kindle for Mac reading app gives users the abi...,Start reading immediately with three free book...,"{""Release date"": ""July 29, 2015"", ""Pricing"": ""..."
5983,b00a4o6nmg,my singing monsters,appstore for android,,Welcome to My Singing Monsters! A world inhabi...,30+ epic and unreal monster species. Collect t...,"{""Release Date"": ""2013"", ""Date first listed on..."
30605,b007zgo7em,calculator plus free,appstore for android,,"*** One of USA TODAY's ""25 essential Kindle Fi...","One of USA TODAY's ""25 essential Kindle Fire a...","{""Release Date"": ""2012"", ""Date first listed on..."
32313,b008jgsm6g,flow free,appstore for android,,Flow Free® is a simple yet addictive puzzle ga...,"Over 2,500 free puzzles,Free Play and Time Tri...","{""Release Date"": ""2012"", ""Date first listed on..."
44137,b00xzfcvk4,max,appstore for android,,"It’s all here. Iconic series, award-winning mo...","Browse or search with ease across HBO, movies,...","{""Release Date"": ""2015"", ""Date first listed on..."


### Save Model

In [7]:
# Save the model typically stores only learned parameters (e.g. latent factors)
joblib.dump(model, "../models/als_model.joblib")

# Save the user-item matrix, defines the mapping between users and items used during training/inference
scipy.sparse.save_npz("../models/cf_user_item_matrix.npz", interaction_matrix)

# Save the index mapping for idx to user_id
with open("../models/idx_to_userid_mapping.json", "w") as f:
    json.dump({str(k): v for k, v in idx_to_userid.items()}, f)

# Save the index mapping for idx to items parent_asin
with open("../models/idx_to_itempasin_mapping.json", "w") as f:
    json.dump({str(k): v for k, v in idx_to_itempasin.items()}, f)


### Load model for inferencing

In [8]:
# Load the CF model
model = joblib.load("../models/als_model.joblib")

# Load the user-item matrix
interaction_matrix = scipy.sparse.load_npz(
    "../models/cf_user_item_matrix.npz"
)

# Load user index mapping (model index → real user_id)
with open("../models/idx_to_userid_mapping.json") as f:
    idx_to_userid = {int(k): v for k, v in json.load(f).items()}

# Load item index mapping (model index → real item parent_asin)
with open("../models/idx_to_itempasin_mapping.json") as f:
    idx_to_itempasin = {int(k): v for k, v in json.load(f).items()}


In [9]:
userid_to_idx = {v: k for k, v in idx_to_userid.items()}

# Recommend
recomm_idx, recomm_scores = model.recommend(
    userid=np.int32(userid_to_idx[target_userid]),
    user_items=interaction_matrix[userid_to_idx[target_userid]],
    N=5
)

recomm_asins = [idx_to_itempasin[i] for i in recomm_idx]

print(f"Recommendations for {target_userid}")

recomm_items = item_df[item_df["parent_asin"].isin(recomm_asins)]

recomm_items[["parent_asin", "item_title", "main_category", "categories", "description", "features", "details"]]

Recommendations for ae2222frpdmnomyomcwiantxp7uq


,parent_asin,item_title,main_category,categories,description,features,details
3633,b00uc7dg6q,kindle for mac [download],software,"education & reference,software,writing & liter...",Kindle for Mac reading app gives users the abi...,Start reading immediately with three free book...,"{""Release date"": ""July 29, 2015"", ""Pricing"": ""..."
5983,b00a4o6nmg,my singing monsters,appstore for android,,Welcome to My Singing Monsters! A world inhabi...,30+ epic and unreal monster species. Collect t...,"{""Release Date"": ""2013"", ""Date first listed on..."
30605,b007zgo7em,calculator plus free,appstore for android,,"*** One of USA TODAY's ""25 essential Kindle Fi...","One of USA TODAY's ""25 essential Kindle Fire a...","{""Release Date"": ""2012"", ""Date first listed on..."
32313,b008jgsm6g,flow free,appstore for android,,Flow Free® is a simple yet addictive puzzle ga...,"Over 2,500 free puzzles,Free Play and Time Tri...","{""Release Date"": ""2012"", ""Date first listed on..."
44137,b00xzfcvk4,max,appstore for android,,"It’s all here. Iconic series, award-winning mo...","Browse or search with ease across HBO, movies,...","{""Release Date"": ""2015"", ""Date first listed on..."
